# **Atividade Prática**
<font size=3>

- **Tema:** Abordagens não-supervisionadas.
- **Prazo de entrega:** 11 de Maio.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

## **Enunciado:**
<font size=3>

Com base no *dataset* `NewsGroups`, disponível no diretório $\text{dataset/}\,$, desenvolva as seguintes tarefas não-supervisionadas. Nas abordagens abaixo:
- Realize a vetorização seguido de SVD a fim de obter os vetores semântico;
- Utilize as ferramentas como Pipeline e RandomizedSearchCV para organizar e afinar hiperparâmetros do pré-processamento e modelo;
- Descreva em uma célula markdown qual o melhor modelo encontrado.

### **1. Questão:**
<font size=3>
    
- Treine o modelo [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) e o avalie com a métrica [`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html). **Descreva** em uma **célula _markdown_** como funciona a métrica;

- Faça um plote bidimensional, utilizando [`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), para a visualização dos *clusters*. Neste caso, o PCA é mais adequado do que utilizar o SVD já que os dados ficarão centralizados no gráfico — possibilitando melhor definição dos *clusters*.

  

#### Definições gerais e imports

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt

from nltk.corpus           import stopwords
from nltk.corpus           import wordnet
from nltk.tokenize         import word_tokenize
from nltk.stem             import WordNetLemmatizer
from sklearn.base          import BaseEstimator
from sklearn.cluster       import KMeans
from sklearn.compose       import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble      import RandomForestClassifier
from sklearn.svm           import SVC

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, normalize

from scipy.stats import randint, uniform

In [2]:
RANDOM_STATE=42 # semente para garantir a reprodutibilidade dos resultados
TEST_SIZE=0.10 # proporção de dados para teste
SVD_COMPONENTS=25 # número de componentes para o SVD
N_ITER=100 # número de iterações para o RandomizedSearchCV
CV=3 # número de folds para o cross-validation

##### Lendo os dados (do disco local)

In [3]:
df = pd.read_csv("./dataset/newsgroups.csv") # estou executando no meu notebook pessoal, não no google colab.

print(df.shape)

df.head()

(2752, 1)


,reviews
0,\n\n\n\n\n\nYour bad English? (See quote abov...
1,"I was a graduate student in the early 1980s, a..."
2,"\nIt doesn't make a whole lot of difference, a..."
3,\n\n\n\n
4,"Yesterday, I went to the Boeing shareholders m..."


In [4]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
        
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)
    text = re.sub(r"\d+", "", text)

    text = " ".join(token for token in text.split() if token not in stop_words)

    text = re.sub(r"\s+", " ", text).strip()
    return text

for i in range(0,4):
    text_antes = df.iloc[i]['reviews']
    texto_depois = preprocess_text(text_antes)
    print(f"TEXTO {i} ANTES.:@@@{text_antes}@@@FINAL")
    print(f"TEXTO {i} DEPOIS:@@@{texto_depois}@@@FINAL")
    print("--------------------------------")



TEXTO 0 ANTES.:@@@





Your bad English?  (See quote above.)



You'd lose that wager, if the supporting argument were part of it.


Did you know that Hitler himself was a devout Christian?  And heterosexual?

--Drywid@@@FINAL
TEXTO 0 DEPOIS:@@@bad english see quote youd lose wager supporting argument part know hitler devout christian heterosexual drywid@@@FINAL
--------------------------------
TEXTO 1 ANTES.:@@@I was a graduate student in the early 1980s, and we had a conference on 
Reaganomics where Jerry Jordan, then a member of the Council of Economic 
Advisors, was a speaker.  I had the pleasure of driving him back to the 
airport afterwards, and since taxes were the main topic of discussion I 
thought I would ask him about the VAT.  I have favored it for these reasons 
you mention, that the income base is too hazy to define, that it taxes 
savings and investment, that it is likely to be more visible.  He agreed, 
and reported that the CEA at that time was in favor of VAT.  So wh

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
df.head()

print(f"ANTES: {df['reviews'].isna().sum()}")
df = df.dropna(subset=["reviews"])
df = df.reset_index(drop=True)
print(f"DEPOIS: {df['reviews'].isna().sum()}")

df.count()


ANTES: 77
DEPOIS: 0


reviews    2675
dtype: int64

In [6]:
# Definindo a coluna e o tipo de transformação:
# 1. texto:
text_atribute = "reviews"
text_pipe = Pipeline(steps=[
    ("vec", TfidfVectorizer(preprocessor=preprocess_text)),
    ("svd", TruncatedSVD())
])

# 2. dados categóricos:
#categorical_atributes = ['Product', 'Brand']
#categorical_transformer = OneHotEncoder()

# 3. dados numéricos:
#numerical_atributes = ['Age']
#numeric_transformer = StandardScaler()

# definindo o objeto ColumnTransformer para aplicar as transformações em X:
preprocessor = ColumnTransformer(transformers=[("text", text_pipe, text_atribute)],
                                 remainder='drop' # colunas que não foram identificadas serão removidas
                                )

pipe = Pipeline(steps=[('preprocessor', preprocessor),
                       ('classifier', KMeans(random_state=RANDOM_STATE, verbose=1))])

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [7]:
def kmeans_silhouette_scorer(estimator, X, y=None):
    X_transformed = estimator.named_steps["preprocessor"].transform(X)
    labels = estimator.named_steps["classifier"].predict(X_transformed)

    if len(set(labels)) < 2:
        return -1

    return silhouette_score(X_transformed, labels)


In [8]:
param_grid = {'preprocessor__text__vec__min_df': [5, 10, 15],
              'preprocessor__text__vec__max_df': [0.80, 0.90, 0.95],
              'preprocessor__text__svd__n_components':[10, 15, 25, 35, 50, 75, 100, 150, 200],
              'classifier__n_clusters': [2, 3, 4, 5, 6, 8, 10]
             }

random_search = RandomizedSearchCV(estimator=pipe,
                                   param_distributions=param_grid,
                                   n_iter=N_ITER,
                                   cv=CV,
                                   scoring=kmeans_silhouette_scorer,
                                   verbose=0, 
                                   random_state=RANDOM_STATE)

random_search

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=1))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__n_clusters': [2, 3, ...], 'preprocessor__text__svd__n_components': [10, 15, ...], 'preprocessor__text__vec__max_df': [0.8, 0.9, ...], 'preprocessor__text__vec__min_df': [5, 10, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",<function kme...t 0x11dfbd170>
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.R

In [9]:
total = np.prod([len(v) for v in param_grid.values()])
print(total)

567


In [10]:
random_search.fit(df)

print("Melhor score:", random_search.best_score_)
print("Melhores parâmetros:")
print(random_search.best_params_)

best_pipe = random_search.best_estimator_

Initialization complete
Iteration 0, inertia 489.8540609682974.
Iteration 1, inertia 344.64831479145397.
Iteration 2, inertia 339.80206145625823.
Iteration 3, inertia 338.1267849411496.
Iteration 4, inertia 337.4499477411604.
Iteration 5, inertia 337.07248729355115.
Iteration 6, inertia 336.7959050689627.
Iteration 7, inertia 336.5554888820347.
Iteration 8, inertia 336.39327872841636.
Iteration 9, inertia 336.29094189919897.
Iteration 10, inertia 336.1807659288067.
Iteration 11, inertia 336.07425204784386.
Iteration 12, inertia 336.0014832123662.
Iteration 13, inertia 335.9461189237268.
Iteration 14, inertia 335.9113846192731.
Iteration 15, inertia 335.8772627577381.
Iteration 16, inertia 335.86431774142346.
Iteration 17, inertia 335.8538533118814.
Iteration 18, inertia 335.850499379187.
Iteration 19, inertia 335.85000451660846.
Converged at iteration 19: strict convergence.
Initialization complete
Iteration 0, inertia 434.1380783739539.
Iteration 1, inertia 350.74725047311176.
Iterati

In [11]:
df["cluster"] = best_pipe.predict(df)

df["cluster"].value_counts()

cluster
1    2249
0     426
Name: count, dtype: int64

In [12]:
for c in sorted(df["cluster"].unique()):
    print(f"\nCLUSTER {c}")
    print(df[df["cluster"] == c]["reviews"].head(10).to_string(index=False))


CLUSTER 0
Yesterday, I went to the Boeing shareholders me...
\n"I hold that space cannot be curved, for the ...
Would someone please send me a list of the hist...
Original to: wats@scicom.AlphaCDC.COM\nG'day wa...
\nActually, Hiten wasn't originally intended to...
The most current orbital elements from the NORA...
\n\n\nThe Command Loss Timer is a timer that do...
Thanks to the people who have answered here and...
on Date: 01 Apr 93 18:03:12 GMT, Ralph Buttigie...
Apologies if this gets posted twice, but I don'...

CLUSTER 1
\n\n\n\n\n\nYour bad English?  (See quote above...
I was a graduate student in the early 1980s, an...
\nIt doesn't make a whole lot of difference, ac...
                                          \n\n\n\n
\n\nAlso would maybe get the Russians Involved....
You guys are correct.  The Bricklin was produce...
\n\nOK.  I stand corrected.  I guess, then, tha...
\n\n\nPity you didn't say something about the u...
\n\n\n\n \n                     ^^^^^^^^^\n\n\n...
From arti

### **2. Questão:**
<font size=3>

Utilize o modelo de Alocação Latente de Dirichlet a fim de realizar a análise de tópicos. Para isso:
- Imprima na tela a quantidade de tópicos que melhor ajustou aos dados;
- Plote as 10 palavras mais associadas a cada tópico;
- Plote 10 documentos com a distribuição de tópicos;
- Escreva em uma célula markdown sua interpretação desses resultados. 